Cell 1 — Mount Drive and load settings

In [8]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:


import os
import json
import yaml
import joblib
import numpy as np
import pandas as pd
import torch

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

PROJECT_ROOT = "/content/drive/MyDrive/atlas-go-revision"
os.chdir(PROJECT_ROOT)

with open("configs/config.yaml", "r") as file:
    cfg = yaml.safe_load(file)

PROCESSED_DIR = cfg["paths"]["data_processed"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


Cell 2 — Load processed data and the homology-aware split

In [10]:
df = pd.read_csv(f"{PROCESSED_DIR}/full_dataset_processed.csv")
Y_all = np.load(f"{PROCESSED_DIR}/Y_all.npy")

splits = np.load(f"{PROCESSED_DIR}/splits_homology.npz")

train_idx = splits["train_idx"]
val_idx = splits["val_idx"]
test_idx = splits["test_idx"]

S_COLS = cfg["data"]["static_features"]
D_COLS = cfg["data"]["dynamic_features"]

PROTBERT_MAX_RESIDUES = cfg["data"]["protbert_max_residues"]
ESM2_MAX_RESIDUES = cfg["data"]["esm2_max_residues"]

print("Proteins:", len(df))
print("GO labels:", Y_all.shape[1])
print("Train / validation / test:", len(train_idx), len(val_idx), len(test_idx))
print("ProtBERT residue limit:", PROTBERT_MAX_RESIDUES)
print("ESM2 residue limit:", ESM2_MAX_RESIDUES)

assert len(df) == Y_all.shape[0]

Proteins: 1356
GO labels: 1961
Train / validation / test: 1091 133 132
ProtBERT residue limit: 1000
ESM2 residue limit: 1000


Cell 3 — Prepare and validate sequences

In [11]:
import re

def clean_sequence(sequence):
    """
    Replace unusual amino-acid letters with X before language-model inference.
    """
    sequence = str(sequence).upper().replace(" ", "")

    # Standard ProtBERT preprocessing.
    sequence = re.sub(r"[UZOB]", "X", sequence)

    # Replace any remaining unexpected character with X.
    sequence = re.sub(r"[^ACDEFGHIKLMNPQRSTVWYX]", "X", sequence)

    return sequence

sequences = [clean_sequence(sequence) for sequence in df["Sequence"]]

if any(len(sequence) == 0 for sequence in sequences):
    raise ValueError("At least one protein sequence is empty.")

print("Sequences ready.")
print("Shortest sequence:", min(map(len, sequences)))
print("Longest sequence:", max(map(len, sequences)))

Sequences ready.
Shortest sequence: 46
Longest sequence: 7176


Cell 4 — Extract ProtBERT embeddings

Cell 4 — Extract ProtBERT embeddings

In [12]:
from transformers import BertTokenizer, BertModel

protbert_path = f"{PROCESSED_DIR}/P_embeddings_raw.npy"

if os.path.exists(protbert_path):
    P_embeddings = np.load(protbert_path)
    print("Loaded existing ProtBERT embeddings:", P_embeddings.shape)

else:
    print("Loading ProtBERT...")
    tokenizer_p = BertTokenizer.from_pretrained(
    "Rostlab/prot_bert",
    do_lower_case=False,
)


    model_p = BertModel.from_pretrained("Rostlab/prot_bert")
    model_p = model_p.to(device)
    model_p.eval()

    PROTBERT_BATCH_SIZE = cfg["feature_extraction"]["protbert_batch_size"]

    def extract_protbert_embeddings(sequences, batch_size):
        """
        Mean-pool ProtBERT residue embeddings.

        Each protein is explicitly limited to 1,000 residues.
        max_length is 1,002 because ProtBERT adds [CLS] and [SEP].
        """
        all_embeddings = []

        for start in tqdm(
            range(0, len(sequences), batch_size),
            desc="ProtBERT batches",
        ):
            batch_sequences = sequences[start:start + batch_size]

            # Limit proteins to the first 1,000 residues.
            spaced_sequences = [
                " ".join(list(sequence[:PROTBERT_MAX_RESIDUES]))
                for sequence in batch_sequences
            ]

            inputs = tokenizer_p(
                spaced_sequences,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=PROTBERT_MAX_RESIDUES + 2,
            )

            inputs = {
                name: value.to(device)
                for name, value in inputs.items()
            }

            with torch.no_grad():
                outputs = model_p(**inputs)

            token_embeddings = outputs.last_hidden_state

            # Build a mask that contains residues only:
            # remove [CLS], [SEP], and padding positions.
            residue_mask = inputs["attention_mask"].clone()

            # Remove [CLS].
            residue_mask[:, 0] = 0

            # Remove [SEP], which is the final non-padding token.
            sequence_token_counts = inputs["attention_mask"].sum(dim=1)
            sep_positions = sequence_token_counts - 1

            residue_mask[
                torch.arange(len(batch_sequences), device=device),
                sep_positions,
            ] = 0

            residue_mask = residue_mask.unsqueeze(-1).float()

            summed_embeddings = (token_embeddings * residue_mask).sum(dim=1)
            residue_counts = residue_mask.sum(dim=1).clamp(min=1.0)

            batch_embeddings = (
                summed_embeddings / residue_counts
            ).cpu().numpy()

            all_embeddings.append(batch_embeddings.astype(np.float32))

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        return np.vstack(all_embeddings)

    P_embeddings = extract_protbert_embeddings(
        sequences=sequences,
        batch_size=PROTBERT_BATCH_SIZE,
    )

    np.save(protbert_path, P_embeddings)

    print("Saved ProtBERT embeddings:", P_embeddings.shape)

    del model_p
    del tokenizer_p

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

assert P_embeddings.shape == (len(df), 1024)

Loading ProtBERT...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/81.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/487 [00:00<?, ?it/s]

BertModel LOAD REPORT from: Rostlab/prot_bert
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ProtBERT batches:   0%|          | 0/85 [00:00<?, ?it/s]

Saved ProtBERT embeddings: (1356, 1024)


Cell 5 — Extract ESM2 embeddings

In [13]:
!pip install fair-esm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 6.2 MB/s eta 0:00:00


In [14]:
import esm

esm2_path = f"{PROCESSED_DIR}/E_embeddings_raw.npy"

if os.path.exists(esm2_path):
    E_embeddings = np.load(esm2_path)
    print("Loaded existing ESM2 embeddings:", E_embeddings.shape)

else:
    print("Loading ESM2 650M...")

    esm_model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
    esm_model = esm_model.to(device)
    esm_model.eval()

    batch_converter = alphabet.get_batch_converter()

    ESM2_BATCH_SIZE = cfg["feature_extraction"]["esm2_batch_size"]

    def extract_esm2_embeddings(sequences, batch_size):
        """
        Mean-pool ESM2 final-layer residue embeddings.

        Each sequence is explicitly limited to 1,000 residues.
        BOS and EOS tokens are excluded from pooling.
        """
        all_embeddings = []

        for start in tqdm(
            range(0, len(sequences), batch_size),
            desc="ESM2 batches",
        ):
            batch_sequences = sequences[start:start + batch_size]

            batch_data = [
                (
                    f"protein_{start + index}",
                    sequence[:ESM2_MAX_RESIDUES],
                )
                for index, sequence in enumerate(batch_sequences)
            ]

            batch_labels, batch_strings, batch_tokens = batch_converter(
                batch_data
            )

            batch_tokens = batch_tokens.to(device)

            with torch.no_grad():
                results = esm_model(
                    batch_tokens,
                    repr_layers=[33],
                    return_contacts=False,
                )

            token_representations = results["representations"][33]

            # ESM2 position 0 is BOS.
            # Residues are positions 1 through sequence_length.
            for index, sequence in enumerate(batch_strings):
                residue_length = len(sequence)

                embedding = (
                    token_representations[index, 1:residue_length + 1]
                    .mean(dim=0)
                    .cpu()
                    .numpy()
                    .astype(np.float32)
                )

                all_embeddings.append(embedding)

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        return np.vstack(all_embeddings)

    E_embeddings = extract_esm2_embeddings(
        sequences=sequences,
        batch_size=ESM2_BATCH_SIZE,
    )

    np.save(esm2_path, E_embeddings)

    print("Saved ESM2 embeddings:", E_embeddings.shape)

    del esm_model
    del alphabet
    del batch_converter

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

assert E_embeddings.shape == (len(df), 1280)

Loading ESM2 650M...
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t33_650M_UR50D.pt" to /root/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t33_650M_UR50D-contact-regression.pt" to /root/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D-contact-regression.pt


ESM2 batches:   0%|          | 0/339 [00:00<?, ?it/s]

Saved ESM2 embeddings: (1356, 1280)


Cell 6 — Normalize ATLAS Static and Dynamic features without leakage

In [15]:
# Static features: no missing values should remain after Notebook 01.
S_raw = df[S_COLS].astype(float).to_numpy()

if np.isnan(S_raw).any():
    raise ValueError("Static ATLAS features contain missing values.")

# Dynamic features can contain missing values.
# Their imputer and scaler are fit on TRAINING proteins only.
D_raw = df[D_COLS].astype(float).to_numpy()

static_scaler = StandardScaler()
S_normalized = np.zeros_like(S_raw, dtype=np.float32)

S_normalized[train_idx] = static_scaler.fit_transform(S_raw[train_idx])
S_normalized[val_idx] = static_scaler.transform(S_raw[val_idx])
S_normalized[test_idx] = static_scaler.transform(S_raw[test_idx])

dynamic_imputer = SimpleImputer(strategy="mean")
D_imputed = np.zeros_like(D_raw, dtype=np.float64)

D_imputed[train_idx] = dynamic_imputer.fit_transform(D_raw[train_idx])
D_imputed[val_idx] = dynamic_imputer.transform(D_raw[val_idx])
D_imputed[test_idx] = dynamic_imputer.transform(D_raw[test_idx])

dynamic_scaler = StandardScaler()
D_normalized = np.zeros_like(D_imputed, dtype=np.float32)

D_normalized[train_idx] = dynamic_scaler.fit_transform(D_imputed[train_idx])
D_normalized[val_idx] = dynamic_scaler.transform(D_imputed[val_idx])
D_normalized[test_idx] = dynamic_scaler.transform(D_imputed[test_idx])

np.save(f"{PROCESSED_DIR}/S_normalized_homology.npy", S_normalized)
np.save(f"{PROCESSED_DIR}/D_normalized_homology.npy", D_normalized)

joblib.dump(
    static_scaler,
    f"{PROCESSED_DIR}/static_scaler_homology.joblib",
)

joblib.dump(
    dynamic_imputer,
    f"{PROCESSED_DIR}/dynamic_imputer_homology.joblib",
)

joblib.dump(
    dynamic_scaler,
    f"{PROCESSED_DIR}/dynamic_scaler_homology.joblib",
)

print("Static features:", S_normalized.shape)
print("Dynamic features:", D_normalized.shape)
print("Scalers were fit only on the training split.")

Static features: (1356, 4)
Dynamic features: (1356, 4)
Scalers were fit only on the training split.


Cell 7 — Build all nine feature sets

In [16]:
feature_sets = {
    "p": P_embeddings,
    "ps": np.hstack([P_embeddings, S_normalized]),
    "psd": np.hstack([P_embeddings, S_normalized, D_normalized]),

    "e": E_embeddings,
    "es": np.hstack([E_embeddings, S_normalized]),
    "esd": np.hstack([E_embeddings, S_normalized, D_normalized]),

    "pe": np.hstack([P_embeddings, E_embeddings]),
    "pes": np.hstack([P_embeddings, E_embeddings, S_normalized]),
    "pesd": np.hstack([
        P_embeddings,
        E_embeddings,
        S_normalized,
        D_normalized,
    ]),
}

expected_dimensions = {
    "p": 1024,
    "ps": 1028,
    "psd": 1032,
    "e": 1280,
    "es": 1284,
    "esd": 1288,
    "pe": 2304,
    "pes": 2308,
    "pesd": 2312,
}

for feature_name, matrix in feature_sets.items():
    matrix = matrix.astype(np.float32)

    if matrix.shape != (len(df), expected_dimensions[feature_name]):
        raise ValueError(
            f"Unexpected shape for {feature_name}: {matrix.shape}"
        )

    output_path = f"{PROCESSED_DIR}/X_{feature_name}_homology.npy"

    np.save(output_path, matrix)

    print(
        f"{feature_name.upper():4s} "
        f"shape={matrix.shape} "
        f"saved={os.path.basename(output_path)}"
    )

P    shape=(1356, 1024) saved=X_p_homology.npy
PS   shape=(1356, 1028) saved=X_ps_homology.npy
PSD  shape=(1356, 1032) saved=X_psd_homology.npy
E    shape=(1356, 1280) saved=X_e_homology.npy
ES   shape=(1356, 1284) saved=X_es_homology.npy
ESD  shape=(1356, 1288) saved=X_esd_homology.npy
PE   shape=(1356, 2304) saved=X_pe_homology.npy
PES  shape=(1356, 2308) saved=X_pes_homology.npy
PESD shape=(1356, 2312) saved=X_pesd_homology.npy


Cell 8 — Save extraction metadata

In [17]:
metadata = {
    "n_proteins": int(len(df)),
    "protbert_model": "Rostlab/prot_bert",
    "protbert_embedding_dimension": 1024,
    "protbert_max_residues": int(PROTBERT_MAX_RESIDUES),
    "protbert_pooling": "mean pooling over residue tokens only; CLS, SEP, and padding excluded",

    "esm2_model": "esm2_t33_650M_UR50D",
    "esm2_embedding_dimension": 1280,
    "esm2_max_residues": int(ESM2_MAX_RESIDUES),
    "esm2_layer": 33,
    "esm2_pooling": "mean pooling over residue tokens only; BOS and EOS excluded",

    "static_features": S_COLS,
    "dynamic_features": D_COLS,
    "normalization": (
        "Static and Dynamic ATLAS preprocessing was fitted only on "
        "training proteins from the homology-aware split."
    ),

    "feature_set_dimensions": expected_dimensions,
}

with open(
    f"{PROCESSED_DIR}/feature_extraction_metadata.json",
    "w",
) as file:
    json.dump(metadata, file, indent=2)

print("Saved feature_extraction_metadata.json")

Saved feature_extraction_metadata.json


Cell 9 — Final verification

In [18]:
print("=" * 60)
print("FEATURE EXTRACTION COMPLETE")
print("=" * 60)

for feature_name, expected_dim in expected_dimensions.items():
    path = f"{PROCESSED_DIR}/X_{feature_name}_homology.npy"
    matrix = np.load(path, mmap_mode="r")

    print(
        f"{feature_name.upper():4s}: "
        f"{matrix.shape[0]} proteins × {matrix.shape[1]} features"
    )

print("\nNext notebook: 03_training_evaluation.ipynb")

FEATURE EXTRACTION COMPLETE
P   : 1356 proteins × 1024 features
PS  : 1356 proteins × 1028 features
PSD : 1356 proteins × 1032 features
E   : 1356 proteins × 1280 features
ES  : 1356 proteins × 1284 features
ESD : 1356 proteins × 1288 features
PE  : 1356 proteins × 2304 features
PES : 1356 proteins × 2308 features
PESD: 1356 proteins × 2312 features

Next notebook: 03_training_evaluation.ipynb
